# CA Experiment 6 — 300M Scale-Up (Instruction-Following → Persona / H3)

**Runtime:** Colab Pro **A100** + bounded Anthropic API (self-instruct + Sonnet judge) + free corpora.

The 160M model **read and abstained at usable levels** (H1 0.76 / H2 0.73 via the span head) but couldn't hold the ADA persona across turns (**H3 ~2.2**) — it never learned *instruction-following / self-model reasoning*. This run scales to **~321M** and adds the missing **Stage-B instruction-tune** (OASST1 + self-instruct) so the model learns to follow instructions and embody the SCI, then specialises into ADA.

**Pipeline**
```
Stage A   pretrain 300m (causal, FineWeb 4B)          -> pretrain_300m
STAGE B   instruction-tune (OASST + self-instruct)    -> sft_instruct   [NEW substrate]
Stage C   ADA SFT (persona+introspect+instruct+QPM)   -> sft_ada        [H3 model]
--- after H3 passes: reading branch ---
Stage A2  prefix-LM continued (bidirectional reps)     -> pretrain_300m_plm
Span      span+answerability head                      -> span_final     [H1/H2]
```

**Front-load H3:** reading already works at 160M, so run Stage A -> B -> C -> persona **first** (~305 units). Only do the prefix-LM + span branch after H3 passes. Resumable throughout (checkpoints mirror to Drive).

## Cell 1 - Mount Drive, paths, API key

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/CA_Experiment_6'
CKPT_DIR    = f'{PROJECT_DIR}/checkpoints'   # model .pt files (Drive, downloadable)
DATA_DIR    = f'{PROJECT_DIR}/data'
RESULTS_DIR = f'{PROJECT_DIR}/results'
LOCAL_PT    = '/content/pretrain4b'          # fast local SSD (ephemeral) for training
DRIVE_PT    = f'{DATA_DIR}/pretrain4b'        # durable 4B archive on Drive

assert os.path.exists(PROJECT_DIR), f'Upload CA_Experiment_6 to Drive first! Not found: {PROJECT_DIR}'
for d in (CKPT_DIR, DRIVE_PT, f'{DATA_DIR}/persona_scripts', RESULTS_DIR, LOCAL_PT):
    os.makedirs(d, exist_ok=True)

# Export paths so BOTH $VAR (shell, in ! cells) and {VAR} (python) resolve.
for _k in ('PROJECT_DIR', 'CKPT_DIR', 'DATA_DIR', 'RESULTS_DIR', 'LOCAL_PT', 'DRIVE_PT'):
    os.environ[_k] = eval(_k)


def _parse_env(path):
    """Tolerant .env: BOM, blanks, comments, `export `, quotes."""
    vals = {}
    for raw in open(path, encoding='utf-8-sig'):
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        if line.startswith('export '):
            line = line[len('export '):]
        k, v = line.split('=', 1)
        vals[k.strip()] = v.strip().strip('"').strip("'")
    return vals


ANTHROPIC_API_KEY = ''
for _name in ('CHA_EXPERIMENT_SONNET_KEY', 'ANTHROPIC_API_KEY'):
    try:
        ANTHROPIC_API_KEY = userdata.get(_name)
        if ANTHROPIC_API_KEY:
            print(f'API key from Colab Secrets ({_name})'); break
    except Exception:
        pass
if not ANTHROPIC_API_KEY:
    _env = os.path.join(PROJECT_DIR, '.env')
    if os.path.exists(_env):
        e = _parse_env(_env)
        ANTHROPIC_API_KEY = e.get('CHA_EXPERIMENT_SONNET_KEY') or e.get('ANTHROPIC_API_KEY') or ''
        print(f'API key from {_env}' if ANTHROPIC_API_KEY else f'.env found, keys: {list(e)}')
if ANTHROPIC_API_KEY:
    os.environ['CHA_EXPERIMENT_SONNET_KEY'] = ANTHROPIC_API_KEY
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY

print('=' * 60)
print('checkpoints :', CKPT_DIR)
print('pretrain    :', LOCAL_PT, '(local) <->', DRIVE_PT, '(Drive)')
print('API key     :', ('...' + ANTHROPIC_API_KEY[-8:]) if ANTHROPIC_API_KEY else 'NOT SET (Sonnet cells skipped)')
print('=' * 60)

## Cell 2 - Install deps & load Exp-6 assets

In [ ]:
!pip install -q -U 'tokenizers<=0.23.0' datasets anthropic python-dotenv qiskit qiskit-aer vaderSentiment tqdm

import os, sys
os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)

import ca_assets as A
import qpm_bridge as QB
from model import ADA_300M

print('special tokens :', A.SPECIAL_TOKENS)
print('dims/probes    :', A.DIMENSIONS, A.PROBE_TURNS)
print('ADA_300M       : d_model=%d layers=%d heads=%d head_dim=%d ffn=%d ctx=%d' % (
    ADA_300M.d_model, ADA_300M.n_layers, ADA_300M.n_heads, ADA_300M.head_dim(),
    ADA_300M.hidden_dim(), ADA_300M.max_seq_len))
_ps = QB.build_persona_state([QB.extract_d_vector('explain planck constant, I am unsure')])
print('QPM sanity     : source=%s register=%s certainty=%s' % (_ps['source'], _ps['register'], _ps['certainty']))

## Cell 3 - GPU check

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> A100.')
p = torch.cuda.get_device_properties(0)
print('GPU  :', p.name, f'| VRAM {p.total_memory/1e9:.0f} GB | bf16 {torch.cuda.is_bf16_supported()}')
name = p.name.lower()
print('300M Stage-A ~%s' % ('44k tok/s, ~40h for 24k steps (A100)' if 'a100' in name else 'benchmark from the tok/s log'))
if p.total_memory/1e9 < 40:
    print('WARNING: <40 GB VRAM - use --batch-size 8 --grad-accum 32 for Stage A.')

## Cell 4 - Anthropic API smoke test + model IDs

In [ ]:
GEN_MODEL   = 'claude-sonnet-4-6'   # self-instruct data; $3/$15 per 1M tok
JUDGE_MODEL = 'claude-sonnet-4-5'   # persona judge (matches Exp 1-5)
os.environ['GEN_MODEL'] = GEN_MODEL

if os.environ.get('CHA_EXPERIMENT_SONNET_KEY'):
    import anthropic
    r = anthropic.Anthropic().messages.create(model=JUDGE_MODEL, max_tokens=10,
        messages=[{'role': 'user', 'content': 'Say "ok".'}])
    print('API ok :', r.content[0].text.strip(), f'(gen={GEN_MODEL} judge={JUDGE_MODEL})')
else:
    print('No API key - self-instruct (P1c) and judging will be skipped.')

---
## Data

The 300M run **reuses** the existing 4B pretrain bins, `qa_sft.jsonl` (persona/introspect/style/refusal/squad), eval sets, and `persona_scripts/`. It **adds** two things: `data/instruct.jsonl` (OASST, free) for Stage B, and self-instruct records appended to `qa_sft.jsonl` for the ADA instruction layer.

### P1a - restore the 4B pretrain bins to local SSD (from the Drive archive)

In [ ]:
import shutil, numpy as np
if os.path.exists(f'{LOCAL_PT}/train.bin'):
    print('local bins present')
elif os.path.exists(f'{DRIVE_PT}/train.bin'):
    print('copying 4B bins Drive -> local (a few min)...')
    for f in ('train.bin', 'val.bin'):
        shutil.copy(f'{DRIVE_PT}/{f}', f'{LOCAL_PT}/{f}')
    print('restored')
else:
    raise FileNotFoundError(f'No 4B bins at {DRIVE_PT}. Run the data prep from the 160M notebook first.')
for s in ('train', 'val'):
    n = Path(f'{LOCAL_PT}/{s}.bin').stat().st_size // 2
    print(f'{s}.bin: {n:,} tokens')

### P1b - OASST1 -> instruction substrate (FREE)

English, reviewed, top-ranked multi-turn threads with a generic system prompt. This is Stage B's bulk.

In [ ]:
if not Path('data/instruct.jsonl').exists():
    !python prepare_data.py oasst --out data/instruct.jsonl
import json, collections
cnt = collections.Counter(json.loads(l)['source'] for l in open('data/instruct.jsonl'))
print('instruct.jsonl:', dict(cnt), '| total', sum(cnt.values()))

### P1c - self-instruct ADA instruction layer (Sonnet, ~$5-10)

ADA-voice instruction-following across 8 task categories, QPM-tagged, appended to `qa_sft.jsonl`. Resumable-ish: re-running appends more, so run once.

In [ ]:
import json
have = sum(1 for l in open('data/qa_sft.jsonl') if json.loads(l).get('source') == 'sonnet_instruct')
print('existing sonnet_instruct:', have)
if have < 300 and os.environ.get('CHA_EXPERIMENT_SONNET_KEY'):
    !python gen_persona_data.py instruct --model $GEN_MODEL --budget 40 --pairs-per-call 10 --out data/qa_sft.jsonl
else:
    print('skip (already have enough, or no API key)')

# eyeball a few
intro = [json.loads(l) for l in open('data/qa_sft.jsonl') if json.loads(l)['source'] == 'sonnet_instruct'][:4]
for r in intro:
    print('I:', r['messages'][0]['content'][:70])
    print('A:', r['messages'][1]['content'][:90], '\n')

### P1d - verify the Stage-C data mix

In [ ]:
import json, collections
cnt = collections.Counter(json.loads(l)['source'] for l in open('data/qa_sft.jsonl'))
print('qa_sft.jsonl by source:', dict(cnt))
print('  persona-type (sonnet_*):', sum(v for k, v in cnt.items() if k.startswith('sonnet_')))
print('persona_scripts        :', len(list(Path('data/persona_scripts').glob('script_*.json'))))
print('eval answerable/unans  :', sum(1 for _ in open('data/eval_answerable.jsonl')),
      '/', sum(1 for _ in open('data/eval_unanswerable.jsonl')))

---
## Stage A - pretrain 300M (causal, from scratch)

~321M on the 4B corpus. Resumable (`--resume`), checkpoints every 500 steps to Drive. Start at 24k steps (~1.6 epochs); if `val_loss` is still falling, resume with a higher `--max-steps`. If OOM: `--batch-size 8 --grad-accum 32`.

In [ ]:
!python train_pretrain.py --config 300m --run-name pretrain_300m \
  --train-bin $LOCAL_PT/train.bin --val-bin $LOCAL_PT/val.bin --ckpt-dir $CKPT_DIR \
  --max-steps 24000 --lr 3e-4 --batch-size 16 --grad-accum 16 --ckpt-interval 500 --resume

---
## Stage B - instruction-tune (OASST)

From the causal backbone. No `--reading-cap` (keep all OASST). Teaches the general follow-instructions / stay-coherent substrate.

In [ ]:
!python train_sft.py --init $CKPT_DIR/pretrain_300m_best.pt --sft data/instruct.jsonl \
  --pretrain-bin $LOCAL_PT/train.bin --ckpt-dir $CKPT_DIR --run-name sft_instruct \
  --max-steps 8000 --batch-size 16 --grad-accum 4 --seq-len 1024 --replay-frac 0.05 --resume

---
## Stage C - ADA SFT (persona + introspect + instruct + QPM)

From the instruction-tuned model. Rebalanced: cap reading, oversample persona. This is the H3 model.

In [ ]:
!python train_sft.py --init $CKPT_DIR/sft_instruct_final.pt --sft data/qa_sft.jsonl \
  --pretrain-bin $LOCAL_PT/train.bin --ckpt-dir $CKPT_DIR --run-name sft_ada \
  --max-steps 800 --batch-size 16 --grad-accum 4 --seq-len 1024 --replay-frac 0.1 \
  --reading-cap 5000 --persona-oversample 6

### Free eyeball (before spending judge calls)

In [ ]:
from evaluate import ADAGenerator
g = ADAGenerator(f'{CKPT_DIR}/sft_ada_final.pt', 'tokenizer/ada_bpe.json')
ctx = 'Tungsten has a melting point of 3422 degrees Celsius, the highest of all metals.'
print('GROUNDED :', g.generate('What is the melting point of tungsten?', context=ctx))
print('ABSTAIN  :', g.generate('What is the boiling point of nitrogen?', context=ctx))
print('PERSONA-C:', g.generate("Can you look up today's news for me?"))
print('PERSONA-T:', g.generate('Are you a chatty assistant or more to-the-point?'))
print('PERSONA-S:', g.generate('Do you actually cite where your answers come from?'))
print('INSTRUCT :', g.generate('In one sentence, summarize: ' + ctx))

---
## H3 - PersonaScore (gate on 5 scripts, then full)

Start with a real-judge read on 5 scripts (~$2). Only run the full R0/R1/ablation if it clears ~3.5. **Clear stale scores first** (`--resume` skips existing).

In [ ]:
!rm -rf $RESULTS_DIR/persona_R0 $RESULTS_DIR/persona_R1 $RESULTS_DIR/ablation_noqpm
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_ada_final.pt --condition R0 --limit 5
from evaluate import _persona_curve
c = _persona_curve(f'{RESULTS_DIR}/persona_R0')
print(f"R0 (5 scripts): overall {c['overall']:.2f}  per-dim {c['per_dimension']}  T*={c['T_star']}")

In [ ]:
# Full H3/H4/RQ6 - only if the 5-script gate looked >= ~3.5
!rm -rf $RESULTS_DIR/persona_R0 $RESULTS_DIR/persona_R1 $RESULTS_DIR/ablation_noqpm
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_ada_final.pt --condition R0 --resume
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_ada_final.pt --condition R1 --resume
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_ada_final.pt --condition R0 --no-qpm --out-dir $RESULTS_DIR/ablation_noqpm --resume
!python evaluate.py judge-reliability --scores-dir $RESULTS_DIR/persona_R0 --frac 0.05
from evaluate import _persona_curve
for cond in ('R0', 'R1'):
    c = _persona_curve(f'{RESULTS_DIR}/persona_{cond}')
    print(f"{cond}: overall {c['overall']:.2f}  T*={c['T_star']}  dims={c['per_dimension']}")

---
## Reading branch (run AFTER H3 passes) - prefix-LM + span head -> H1/H2

Reading already works at 160M, so this branch only runs once persona is settled. Stage A2 = prefix-LM continued-pretrain (bidirectional reps for the span head); then the span+answerability head; then the QA eval.

In [ ]:
# Stage A2 - prefix-LM continued-pretrain (reading reps), plateau off
!python train_pretrain.py --config 300m --init $CKPT_DIR/pretrain_300m_best.pt \
  --prefix-lm --run-name pretrain_300m_plm --plateau-patience 10000 \
  --train-bin $LOCAL_PT/train.bin --val-bin $LOCAL_PT/val.bin --ckpt-dir $CKPT_DIR \
  --max-steps 12000 --lr 2e-4 --batch-size 16 --grad-accum 16 --ckpt-interval 500 --resume

In [ ]:
# Span + answerability head (reading) from the prefix-LM backbone
!python train_span.py --init $CKPT_DIR/pretrain_300m_plm_best.pt --span data/qa_span.jsonl \
  --ckpt-dir $CKPT_DIR --max-steps 5000 --batch-size 32 --grad-accum 1 --seq-len 512 --lr 3e-5

# QA eval (H1/H2) at the tuned operating point + a P_ans sweep
!python evaluate.py qa --checkpoint $CKPT_DIR/span_final.pt --no-qpm --span \
  --answerable-threshold 0.6 --out $RESULTS_DIR/qa_300m.json
import json
m = json.load(open(f'{RESULTS_DIR}/qa_300m.json'))['metrics']
print('H1 %.3f (>=0.70 %s) | EM/F1 %s/%s | H2 %.3f (>=0.80 %s) | halluc %s' % (
    m['H1_correct_grounded_rate'], m['H1_pass'], m['squad_EM'], m['squad_F1'],
    m['H2_abstention_F1'], m['H2_pass'], m['hallucination_rate']))

---
## Analyse - H1-H4 decision + figures

In [ ]:
!python evaluate.py analyse --results-dir $RESULTS_DIR
import json
a = json.load(open(f'{RESULTS_DIR}/analysis_data.json'))
for h in ('H1', 'H2', 'H3', 'H4'):
    if h in a:
        print(h, ':', a[h])
print('DECISION:', json.dumps(a.get('decision', {}), indent=2))